In [2]:
import argparse
import io

import pandas as pd
import requests # Moved import to the top

BASE = "https://asianbondsonline.adb.org/downloads/standard_download_csv.php"
SERIES = {
    "Int_rate_spread_2yrvs10yr": "vn_2s10s_spread_bps.csv",
    "Yld_volatility": "vn_10y_yield_volatility.csv",
}


def parse_abo_csv(text):
    """ABO CSVs start with a metadata block; data begins at the 'Economy,Date,...' line."""
    lines = text.splitlines()
    start = next(i for i, l in enumerate(lines) if l.startswith("Economy,Date"))
    df = pd.read_csv(io.StringIO("\n".join(lines[start:])))
    df = df.rename(columns={df.columns[1]: "date", df.columns[2]: "value"})
    df["date"] = pd.to_datetime(df["date"])
    return df[["date", "value"]].dropna().sort_values("date")


def fetch(code, economy="VN", years=range(2006, 2027)):
    params = {
        "code": code, "economies": economy,
        "years": ",".join(str(y) for y in years),
        "frequency": "Daily", "add_opts": "", "frequency_switch": "",
    }
    headers = {"User-Agent": "Mozilla/5.0", "Referer": "https://asianbondsonline.adb.org/data-portal/"}
    r = requests.get(BASE, params=params, headers=headers, timeout=60)
    r.raise_for_status()
    if "Economy,Date" not in r.text:
        raise RuntimeError("Unexpected response (likely bot protection). Download the CSV manually.")
    return parse_abo_csv(r.text)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--file", help="parse a CSV you downloaded manually instead of fetching")
    # When running in Colab, argparse gets kernel-specific arguments like -f.
    # Passing an empty list makes it ignore these and allows the script to run.
    a = ap.parse_args([])
    if a.file:
        with open(a.file, encoding="utf-8", errors="ignore") as f:
            df = parse_abo_csv(f.read())
        out = a.file.rsplit(".", 1)[0] + "_clean.csv"
        df.to_csv(out, index=False)
        print(f"saved {out}: {len(df)} rows, {df.date.min():%Y-%m-%d} to {df.date.max():%Y-%m-%d}")
    else:
        for code, out in SERIES.items():
            df = fetch(code)
            df.to_csv(out, index=False)
            print(f"saved {out}: {len(df)} rows, {df.date.min():%Y-%m-%d} to {df.date.max():%Y-%m-%d}")

saved vn_2s10s_spread_bps.csv: 5500 rows, 2006-07-18 to 2026-09-30
saved vn_10y_yield_volatility.csv: 5189 rows, 2006-07-18 to 2026-09-30


In [4]:
import argparse
import itertools
import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")
OUT = "output"


# ---------------------------------------------------------------- data
def load_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"], dayfirst=False)
    s = df.set_index("date")["value"].astype(float).sort_index()
    return s


def load_fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df = pd.read_csv(url)
    df.columns = ["date", "value"]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.dropna().set_index("date")["value"]


def demo_series():
    """Synthetic monthly yield: random walk with drift and a 2026 shock."""
    rng = np.random.default_rng(42)
    idx = pd.date_range("2016-01-31", "2026-09-30", freq="ME")
    shocks = rng.normal(0, 0.12, len(idx))
    shocks[idx >= "2026-03-31"] += 0.03  # war-period upward drift
    return pd.Series(4.5 + np.cumsum(shocks), index=idx)


def to_monthly(s):
    """Month-end average: smooths daily liquidity spikes (e.g. quarter-end)."""
    return s.resample("ME").mean().dropna()


# ---------------------------------------------------------------- tests
def stationarity(x, label):
    adf_stat, adf_p, *_ = adfuller(x, autolag="AIC")
    try:
        kpss_stat, kpss_p, *_ = kpss(x, regression="c", nlags="auto")
    except Exception:
        kpss_p = np.nan
    # ADF H0: unit root. KPSS H0: stationary.
    verdict = "stationary" if (adf_p < 0.05 and (np.isnan(kpss_p) or kpss_p > 0.05)) \
        else "non-stationary" if (adf_p >= 0.05 and kpss_p <= 0.05) else "inconclusive"
    return f"{label:<22} ADF p={adf_p:6.3f} | KPSS p={kpss_p:6.3f} -> {verdict}", verdict


def choose_d(y, report):
    line, v0 = stationarity(y, "level")
    report.append(line)
    line, v1 = stationarity(y.diff().dropna(), "1st difference")
    report.append(line)
    if v0 == "stationary":
        return 0
    if v1 == "stationary":
        return 1
    line, _ = stationarity(y.diff().diff().dropna(), "2nd difference")
    report.append(line)
    return 1  # rates are rarely I(2); keep d=1 and flag it


def select_arima(y, d, max_p=3, max_q=3):
    best = None
    rows = []
    for p, q in itertools.product(range(max_p + 1), range(max_q + 1)):
        try:
            res = ARIMA(y, order=(p, d, q), trend="n" if d else "c").fit()
            rows.append((p, d, q, res.aic))
            if best is None or res.aic < best[1].aic:
                best = ((p, d, q), res)
        except Exception:
            continue
    table = pd.DataFrame(rows, columns=["p", "d", "q", "AIC"]).sort_values("AIC")
    return best, table


def momentum(y):
    """TSMOM-style read on a rate level.
    3M and 12M changes in bp; signal = sign of 12M change, strength confirmed
    if the 3M change agrees (trend still accelerating in the same direction)."""
    ch3 = (y.iloc[-1] - y.iloc[-4]) * 100 if len(y) > 3 else np.nan
    ch12 = (y.iloc[-1] - y.iloc[-13]) * 100 if len(y) > 12 else np.nan
    sig = np.sign(ch12)
    agree = np.sign(ch3) == sig
    label = {1: "UP-trend (rates rising)", -1: "DOWN-trend (rates falling)", 0: "flat"}[int(sig)]
    return ch3, ch12, label, agree


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--csv")
    g.add_argument("--fred")
    g.add_argument("--demo", action="store_true")
    ap.add_argument("--name", default="Rate (%)")
    ap.add_argument("--horizon", type=int, default=6)
    ap.add_argument("--start", default="2015-01-01")
    # Pass explicit arguments to argparse for Colab execution
    a = ap.parse_args(['--csv', 'vn_2s10s_spread_bps.csv', '--name', 'Vietnam 2Y-10Y Spread (bps)'])

    os.makedirs(OUT, exist_ok=True)
    raw = load_csv(a.csv) if a.csv else load_fred(a.fred) if a.fred else demo_series()
    name = "DEMO synthetic yield (%)" if a.demo else a.name
    y = to_monthly(raw[raw.index >= a.start])
    y.index = y.index.to_period("M").to_timestamp("M")
    y = y.asfreq("ME")
    report = [f"Series: {name}", f"Sample: {y.index[0]:%Y-%m} to {y.index[-1]:%Y-%m} "
              f"({len(y)} monthly obs)", ""]
    if len(y) < 48:
        report.append("WARNING: fewer than 48 monthly observations; ARIMA "
                      "estimates and CIs will be unreliable.")

    # 1. time plot
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(y.index, y.values, lw=1.6)
    ax.set_title(f"{name}: monthly average")
    ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{OUT}/1_timeplot.png", dpi=160); plt.close(fig)

    # 2. stationarity + transformation
    report.append("Stationarity (ADF H0 = unit root; KPSS H0 = stationary)")
    d = choose_d(y, report)
    report.append(f"-> chosen differencing order d = {d}\n")

    # 3. ACF / PACF of the transformed series
    z = y.diff(d).dropna() if d else y
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    lags = min(24, len(z) // 2 - 1)
    plot_acf(z, lags=lags, ax=axes[0]); plot_pacf(z, lags=lags, ax=axes[1], method="ywm")
    axes[0].set_title(f"ACF (d={d})"); axes[1].set_title(f"PACF (d={d})")
    fig.tight_layout(); fig.savefig(f"{OUT}/2_acf_pacf.png", dpi=160); plt.close(fig)

    # 4. ARIMA by AIC
    (order, res), table = select_arima(y, d)
    report.append("Top 5 ARIMA candidates by AIC:")
    report.append(table.head(5).to_string(index=False))
    report.append(f"-> selected ARIMA{order}\n")
    lb = res.test_serial_correlation("ljungbox", lags=[12])[0]
    report.append(f"Residual Ljung-Box(12) p = {lb[1][-1]:.3f} "
                  f"({'OK, residuals ~ white noise' if lb[1][-1] > 0.05 else 'residual autocorrelation remains'})")

    # 5. forecast
    fc = res.get_forecast(a.horizon)
    mean, ci = fc.predicted_mean, fc.conf_int(alpha=0.05)
    report.append(f"\n{a.horizon}-month forecast (95% CI):")
    for t in mean.index:
        report.append(f"  {t:%Y-%m}: {mean[t]:.2f}  [{ci.loc[t].iloc[0]:.2f}, {ci.loc[t].iloc[1]:.2f}]")
    width = (ci.iloc[-1, 1] - ci.iloc[-1, 0]) * 100
    move = (mean.iloc[-1] - y.iloc[-1]) * 100
    report.append(f"Point move over horizon: {move:+.0f} bp; CI width at horizon: {width:.0f} bp")
    if abs(move) < width / 4:
        report.append("READ: forecast change is small relative to uncertainty -> ARIMA alone "
                      "does NOT support a directional call; lean on momentum and fundamentals.")

    hist = y[y.index >= y.index[-1] - pd.DateOffset(years=4)]
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(hist.index, hist.values, lw=1.6, label="actual")
    ax.plot(mean.index, mean.values, "--", lw=1.6, label=f"ARIMA{order} forecast")
    ax.fill_between(mean.index, ci.iloc[:, 0], ci.iloc[:, 1], alpha=.2, label="95% CI")
    ax.set_title(f"{name}: {a.horizon}-month ARIMA forecast"); ax.legend(); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f"{OUT}/3_forecast.png", dpi=160); plt.close(fig)

    # 6. momentum
    ch3, ch12, label, agree = momentum(y)
    report.append(f"\nTime-series momentum: 3M change {ch3:+.0f} bp | 12M change {ch12:+.0f} bp")
    report.append(f"Signal: {label}; 3M {'confirms' if agree else 'diverges from'} 12M "
                  f"({'trend intact' if agree else 'possible turning point'})")

    text = "\n".join(report)
    with open(f"{OUT}/report.txt", "w") as f:
        f.write(text)
    print(text)


if __name__ == "__main__":
    main()

Series: Vietnam 2Y-10Y Spread (bps)
Sample: 2015-01 to 2026-09 (141 monthly obs)

Stationarity (ADF H0 = unit root; KPSS H0 = stationary)
level                  ADF p= 0.089 | KPSS p= 0.010 -> non-stationary
1st difference         ADF p= 0.000 | KPSS p= 0.100 -> stationary
-> chosen differencing order d = 1

Top 5 ARIMA candidates by AIC:
 p  d  q         AIC
 2  1  1 1211.250811
 1  1  2 1212.460360
 1  1  3 1212.593155
 3  1  1 1212.887137
 2  1  2 1213.250287
-> selected ARIMA(2, 1, 1)

Residual Ljung-Box(12) p = 0.873 (OK, residuals ~ white noise)

6-month forecast (95% CI):
  2026-10: 84.52  [49.70, 119.34]
  2026-11: 86.25  [29.80, 142.69]
  2026-12: 87.23  [17.12, 157.33]
  2027-01: 87.62  [9.27, 165.97]
  2027-02: 87.67  [4.10, 171.24]
  2027-03: 87.57  [0.25, 174.89]
Point move over horizon: +530 bp; CI width at horizon: 17464 bp
READ: forecast change is small relative to uncertainty -> ARIMA alone does NOT support a directional call; lean on momentum and fundamentals.

Time-s